In [2]:
from jupyter_dash import JupyterDash
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output
import pandas as pd
from CRUD_Python_Module import AnimalShelter

JupyterDash.infer_jupyter_proxy_config()

username = "admin"
password = "Admin123!"
db = AnimalShelter(username, password)

df = pd.DataFrame.from_records(db.read({}))
df.drop(columns=['_id'], inplace=True)

app = JupyterDash(__name__)

app.layout = html.Div([
    html.Center(html.B(html.H1('CS-340 Dashboard - Paige Marinkov'))),
    html.Hr(),

    dcc.RadioItems(
        id='filter-type',
        options=[
            {'label': 'All', 'value': 'RESET'},
            {'label': 'Water Rescue', 'value': 'WATER'},
            {'label': 'Mountain or Wilderness Rescue', 'value': 'MOUNTAIN'},
            {'label': 'Disaster or Individual Tracking', 'value': 'DISASTER'}
        ],
        value='RESET',
        labelStyle={'display': 'inline-block', 'margin-right': '20px'}
    ),

    html.Hr(),

    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i} for i in df.columns],
        data=df.to_dict('records'),
        page_size=10,
        sort_action="native",
        filter_action="native",
        row_selectable="single"
    ),

    html.Br(),
    html.Hr(),

    html.Div(style={'display': 'flex'}, children=[
        html.Div(id='graph-id', style={'width': '50%'}),
        html.Div(id='map-id', style={'width': '50%'})
    ])
])


@app.callback(
    Output('datatable-id', 'data'),
    Input('filter-type', 'value')
)
def update_dashboard(filter_type):
    if filter_type == 'WATER':
        filtered_df = df[
            (df['animal_type'] == 'Dog') &
            (df['breed'].str.contains('Labrador Retriever|Chesapeake Bay Retriever|Newfoundland', case=False, na=False)) &
            (df['sex_upon_outcome'].str.contains('Intact Female', case=False, na=False)) &
            (df['age_upon_outcome_in_weeks'] >= 26) &
            (df['age_upon_outcome_in_weeks'] <= 156)
        ]
    elif filter_type == 'MOUNTAIN':
        filtered_df = df[
            (df['animal_type'] == 'Dog') &
            (df['breed'].str.contains('German Shepherd|Alaskan Malamute|Old English Sheepdog|Siberian Husky|Rottweiler', case=False, na=False)) &
            (df['sex_upon_outcome'].str.contains('Intact Male', case=False, na=False)) &
            (df['age_upon_outcome_in_weeks'] >= 26) &
            (df['age_upon_outcome_in_weeks'] <= 156)
        ]
    elif filter_type == 'DISASTER':
        filtered_df = df[
            (df['animal_type'] == 'Dog') &
            (df['breed'].str.contains('Doberman Pinscher|German Shepherd|Golden Retriever|Bloodhound|Rottweiler', case=False, na=False)) &
            (df['sex_upon_outcome'].str.contains('Intact Male', case=False, na=False)) &
            (df['age_upon_outcome_in_weeks'] >= 20) &
            (df['age_upon_outcome_in_weeks'] <= 300)
        ]
    else:
        filtered_df = df

    return filtered_df.to_dict('records')


@app.callback(
    Output('graph-id', "children"),
    Input('datatable-id', "derived_virtual_data")
)
def update_graphs(viewData):
    if viewData is None:
        dff = df
    else:
        dff = pd.DataFrame.from_dict(viewData)

    return [
        dcc.Graph(
            figure=px.pie(dff, names='breed', title='Preferred Animals')
        )
    ]


@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    Input('datatable-id', 'selected_columns')
)
def update_styles(selected_columns):
    if not selected_columns:
        return []
    return [
        {'if': {'column_id': col}, 'background_color': '#D2F3FF'}
        for col in selected_columns
    ]


@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")]
)
def update_map(viewData, index):
    if viewData is None:
        dff = df
    else:
        dff = pd.DataFrame.from_dict(viewData)

    if index is None or len(index) == 0:
        row = 0
    else:
        row = index[0]

    return [
        dl.Map(style={'width': '500px', 'height': '500px'}, center=[30.75, -97.48], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            dl.Marker(position=[dff.iloc[row, 13], dff.iloc[row, 14]], children=[
                dl.Tooltip(dff.iloc[row, 4]),
                dl.Popup([
                    html.H1("Animal Name"),
                    html.P(dff.iloc[row, 9])
                ])
            ])
        ])
    ]


app.run_server(mode='jupyterlab')